# Research duplicate review benchmark — synthetic pilot
**Harder challenge dataset.** Identical titles can belong to independent studies. Exact title matches still bypass thresholds in this original pipeline, deliberately exposing that policy’s false alerts. They are candidates for review, not proven duplicates. Some title-only cases are inherently ambiguous. Warm timing uses three repetitions for a pilot; increase REPS for a final run.

Run in Google Colab using **Runtime → Run all**. Upload the three supplied CSV files when prompted. CPU runtime is sufficient; GPU is optional. No GitHub or Drive connection is needed.

Compares A: PostgreSQL trigram + GIN; B: same candidates ranked by TF-IDF cosine; C: same candidates ranked by Sentence-BERT cosine; D: union of trigram and semantic candidates ranked by Sentence-BERT cosine.

**Synthetic evidence only:** 480 corpus records, 840 queries, and 403,200 explicitly labeled pairs. Duplicate queries are fictional alternate representations of the same study, not judgments inferred from title resemblance. Related-but-distinct queries are negative controls. Vocabulary, templates and labels are deliberately artificial; results cannot establish real-world accuracy. Development/test sources are disjoint, but templates and topics overlap, making this a pilot rather than an independent generalization test. Potential overlap is not labeled or evaluated here.

An exact normalized-title check is shared by all variants. Remaining candidates are ranked by each method; thresholds are calibrated separately on development queries. This challenge defaults to title + abstract; change TEXT_MODE and rerun all downstream cells for title + abstract. Local PostgreSQL storage disappears on runtime reset. Download outputs at the end.

In [ ]:
# Run on a fresh Colab runtime before importing benchmark libraries.
!apt-get -qq update
!apt-get -qq install -y postgresql postgresql-contrib
%pip install -q --upgrade sentence-transformers==6.2.0 transformers==5.22.2 huggingface-hub==1.32.0 psycopg2-binary==2.9.10 tabulate==0.9.0
!service postgresql start
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'colab_synthetic_only';"


In [ ]:
from google.colab import files
uploaded = files.upload()  # Select records.csv, queries.csv and labels.csv together.
import os, json, time, random, platform, hashlib, importlib.metadata, zipfile
import numpy as np
import pandas as pd
import psycopg2
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from huggingface_hub import model_info
from pathlib import Path
records = pd.read_csv('records.csv').fillna('')
queries = pd.read_csv('queries.csv').fillna('')
labels = pd.read_csv('labels.csv')
assert records.id.is_unique and queries.query_id.is_unique
assert len(labels)==len(records)*len(queries)
assert not labels.duplicated(['query_id','record_id']).any()
assert set(labels.label)<= {'duplicate','related_distinct','unrelated'}
assert set(queries[queries.split=='dev'].source_id).isdisjoint(queries[queries.split=='test'].source_id)
TEXT_MODE = 'title_abstract'  # or 'title_abstract'
K_VALUES = [20, 50, 100]
REPS = 3  # pilot timing; use 30 for a longer run
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
output = Path('/content/duplicate_benchmark_results'); output.mkdir(exist_ok=True)
display(queries.groupby(['split','category']).size().rename('count').reset_index())


In [ ]:
conn = psycopg2.connect(dbname='postgres', user='postgres', password='colab_synthetic_only', host='localhost')
conn.autocommit = True
cur = conn.cursor()
# Dedicated disposable Colab table, no connection to application data.
cur.execute('CREATE EXTENSION IF NOT EXISTS pg_trgm')
cur.execute('DROP TABLE IF EXISTS synthetic_research')
cur.execute("CREATE FUNCTION pg_temp.norm(t text) RETURNS text LANGUAGE sql IMMUTABLE AS $$ SELECT trim(regexp_replace(lower(normalize(t,NFKC)), '[^[:alnum:]]+', ' ', 'g')) $$")
cur.execute('CREATE TABLE synthetic_research(id text PRIMARY KEY,title text,normalized_title text)')
from psycopg2.extras import execute_values
execute_values(cur, 'INSERT INTO synthetic_research VALUES %s', [(r.id,r.title,'') for r in records.itertuples()])
cur.execute('UPDATE synthetic_research SET normalized_title=pg_temp.norm(title)')
t0=time.perf_counter()
cur.execute('CREATE INDEX synthetic_trgm ON synthetic_research USING gin(normalized_title gin_trgm_ops)')
index_build_s=time.perf_counter()-t0
cur.execute('ANALYZE synthetic_research')
cur.execute("SET pg_trgm.similarity_threshold=0.3")
cur.execute("SELECT pg_relation_size('synthetic_trgm')")
index_bytes=cur.fetchone()[0]
SQL = """SELECT id, similarity(normalized_title,pg_temp.norm(%s)) AS score,
 normalized_title=pg_temp.norm(%s) AS exact FROM synthetic_research
 WHERE normalized_title %% pg_temp.norm(%s) OR normalized_title LIKE '%%'||pg_temp.norm(%s)||'%%'
 ORDER BY exact DESC, score DESC,id ASC LIMIT %s"""
id_to_idx={v:i for i,v in enumerate(records.id)}
def retrieve(title,k):
    cur.execute(SQL,(title,title,title,title,k))
    return cur.fetchall()
cur.execute('EXPLAIN (ANALYZE, BUFFERS, FORMAT JSON) '+SQL,(queries.title.iloc[0],)*4+(50,))
(output/'query_plan.json').write_text(json.dumps(cur.fetchone()[0],indent=2))
print('Index bytes:',index_bytes,'build seconds:',index_build_s)
print('Small tables may legitimately use a sequential scan; inspect query_plan.json. Do not force an index and claim production speed.')


In [ ]:
def text_column(frame):
    return frame.title.tolist() if TEXT_MODE=='title' else (frame.title+' '+frame.abstract).tolist()
start=time.perf_counter()
tfidf=TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True)
doc_tf=tfidf.fit_transform(text_column(records))  # fit on corpus only, never queries
TF_PREP_S=time.perf_counter()-start
MODEL='sentence-transformers/all-MiniLM-L6-v2'
REVISION=model_info(MODEL).sha  # resolved revision is recorded for repeatability
start=time.perf_counter()
model=SentenceTransformer(MODEL,revision=REVISION,device='cpu')
MODEL_LOAD_S=time.perf_counter()-start
start=time.perf_counter()
doc_emb=model.encode(text_column(records),normalize_embeddings=True,show_progress_bar=True)
EMBED_PREP_S=time.perf_counter()-start
# Global semantic retrieval uses an exact full-corpus cosine scan for this small pilot.
# It is not a vector-index scalability benchmark.
def run(q,method,k):
    start=time.perf_counter()
    candidates=retrieve(q.title,k)
    sql_ms=(time.perf_counter()-start)*1000
    trig={rid:float(score) for rid,score,exact in candidates}
    exact={rid for rid,score,e in candidates if e}
    ids=list(trig)
    text=q.title if TEXT_MODE=='title' else q.title+' '+q.abstract
    if method in ('C','D'):
        qe=model.encode([text],normalize_embeddings=True)[0]
        if method=='D':
            all_scores=doc_emb@qe
            semantic=sorted(range(len(records)),key=lambda i:(-float(all_scores[i]),records.id.iloc[i]))[:k]
            ids=sorted(set(ids)|{records.id.iloc[i] for i in semantic})
        scores=doc_emb[[id_to_idx[r] for r in ids]]@qe if ids else []
    elif method=='B':
        scores=cosine_similarity(tfidf.transform([text]),doc_tf[[id_to_idx[r] for r in ids]]).ravel() if ids else []
    else:
        scores=[trig[r] for r in ids]
    ranked=sorted(zip(ids,map(float,scores)),key=lambda x:(x[0] not in exact,-x[1],x[0]))
    return ranked,exact,sql_ms,(time.perf_counter()-start)*1000
# Correctness checks, outside timing.
exact_queries = queries[queries.category.isin(['exact', 'ambiguous_exact'])]
assert not exact_queries.empty, 'No exact-title test query found; check the uploaded queries.csv.'
q = next(exact_queries.itertuples())
expected_id = getattr(q, 'duplicate_record_id', q.source_id)
assert expected_id in id_to_idx, 'Expected record is missing; upload matching CSV files.'
for method in 'ABCD':
    ranked, exact, _, _ = run(q, method, 50)
    assert expected_id in exact, 'Known duplicate must be an exact-title candidate.'
    assert ranked and ranked[0][0] in exact, 'Exact-title candidates should rank first.'
    # Independent studies can share a title, so the known duplicate need not rank first.
assert np.isfinite(doc_emb).all()
print('Preparation complete:',TEXT_MODE,MODEL,REVISION)


## Run measurements
Quality is deterministic and collected once per query/configuration. Timings repeat with shuffled method order after warmup. Query encoding and local SQL calls are included; document-vector preparation and model loading are recorded separately. This is single-process, warm, CPU execution—not concurrent load testing. Python/service memory is not isolated or reported as peak memory. D may score up to 2K candidates; its higher retrieval budget is intentional and must be disclosed.

In [ ]:
rank_rows=[]; timing=[]; candidate_rows=[]
truth={(r.query_id,r.record_id):r.label for r in labels.itertuples()}
for method in 'ABCD':
    for _ in range(3): run(next(queries.itertuples()),method,50)
for k in K_VALUES:
    for q in queries.itertuples():
        for rep in range(REPS):
            methods=list('ABCD'); random.shuffle(methods)
            for method in methods:
                ranked,exact,sql_ms,total_ms=run(q,method,k)
                timing.append(dict(query_id=q.query_id,split=q.split,method=method,k=k,rep=rep,sql_ms=sql_ms,total_ms=total_ms))
                if rep==0:
                    candidate_rows.append(dict(query_id=q.query_id,split=q.split,category=q.category,method=method,k=k,candidate_count=len(ranked),duplicate_candidates=sum(truth[(q.query_id,r)]=='duplicate' for r,s in ranked)))
                    for rank,(rid,score) in enumerate(ranked[:5],1):
                        rank_rows.append(dict(query_id=q.query_id,split=q.split,category=q.category,method=method,k=k,record_id=rid,rank=rank,score=score,exact=rid in exact,label=truth[(q.query_id,rid)]))
ranks=pd.DataFrame(rank_rows); timings=pd.DataFrame(timing); candidates=pd.DataFrame(candidate_rows)
print('Finished',len(timings),'timed calls')


In [ ]:
# Calibrate an alert threshold on development data only. Exact matches always alert.
# Optimize pair-level F1 over the top-five review shortlist; break ties toward higher thresholds.
thresholds={}; metrics=[]; alert_frames=[]
def summarize(alerts,query_subset):
    tp=int((alerts.label=='duplicate').sum()); fp=len(alerts)-tp
    total=int(labels[labels.query_id.isin(query_subset.query_id)].label.eq('duplicate').sum())
    precision=tp/len(alerts) if len(alerts) else 0.
    recall=tp/total if total else 0.
    return dict(precision=precision,recall=recall,f1=2*precision*recall/(precision+recall) if precision+recall else 0.,false_alerts_per_query=fp/len(query_subset))
for k in K_VALUES:
 for method in 'ABCD':
    dev=ranks[(ranks.split=='dev')&(ranks.method==method)&(ranks.k==k)]
    options=np.r_[[-np.inf],dev.score.unique(),[np.inf]]
    best=max(((summarize(dev[dev.exact|(dev.score>=t)],queries[queries.split=='dev'])['f1'],t) for t in options))
    threshold=best[1];thresholds[f'{method}_{k}']=float(threshold)
    test=ranks[(ranks.split=='test')&(ranks.method==method)&(ranks.k==k)]
    alerts=test[test.exact|(test.score>=threshold)].copy();alert_frames.append(alerts)
    result=summarize(alerts,queries[queries.split=='test'])
    total=int(labels[labels.query_id.isin(queries[queries.split=='test'].query_id)].label.eq('duplicate').sum())
    cs=candidates[(candidates.split=='test')&(candidates.method==method)&(candidates.k==k)]
    ts=timings[(timings.split=='test')&(timings.method==method)&(timings.k==k)]
    result.update(method=method,k=k,candidate_recall=cs.duplicate_candidates.sum()/total,recall_at_5=int(test.label.eq('duplicate').sum())/total,p50_ms=ts.total_ms.median(),p95_ms=ts.total_ms.quantile(.95),threshold=threshold)
    metrics.append(result)
summary=pd.DataFrame(metrics)
display(summary.round(4))
# Paired bootstrap of per-positive-query recall@5 differences at K=50.
rng=np.random.default_rng(SEED)
positive_ids=sorted(set(labels[labels.label=='duplicate'].query_id) & set(queries[queries.split=='test'].query_id))
hits={m:ranks[(ranks.split=='test')&(ranks.k==50)&(ranks.method==m)&(ranks.label=='duplicate')].groupby('query_id').size().reindex(positive_ids,fill_value=0).clip(upper=1).to_numpy() for m in 'ABCD'}
ci=[]
for m in 'BCD':
    delta=hits[m]-hits['A'];boot=np.array([rng.choice(delta,len(delta),replace=True).mean() for _ in range(2000)])
    ci.append(dict(comparison=m+' minus A',recall5_difference=delta.mean(),ci_low=np.quantile(boot,.025),ci_high=np.quantile(boot,.975)))
display(pd.DataFrame(ci))
print('Bootstrap intervals resample templated queries; they do not represent uncertainty on real studies.')


In [ ]:
summary[summary.k==50].set_index('method')[['precision','recall_at_5','candidate_recall']].plot.bar(ylim=(0,1.05),title='Synthetic duplicate detection — K=50')
plt.tight_layout();plt.savefig(output/'quality.png');plt.show()
summary[summary.k==50].set_index('method')[['p50_ms','p95_ms']].plot.bar(title='Warm CPU latency — K=50',ylabel='milliseconds')
plt.tight_layout();plt.savefig(output/'latency.png');plt.show()
alerts=pd.concat(alert_frames,ignore_index=True)
# False-alert breakdown helps identify related-study confusion.
display(alerts[alerts.label!='duplicate'].groupby(['method','k','category','label']).size().rename('false_alerts').reset_index())
for name,frame in [('summary',summary),('ranked_results',ranks),('timings',timings),('candidate_coverage',candidates),('alerts',alerts),('bootstrap',pd.DataFrame(ci))]:
    frame.to_csv(output/f'{name}.csv',index=False)
metadata=dict(text_mode=TEXT_MODE,model=MODEL,revision=REVISION,seed=SEED,reps=REPS,python=platform.python_version(),platform=platform.platform(),cpu=platform.processor(),index_bytes=index_bytes,index_build_s=index_build_s,tfidf_prep_s=TF_PREP_S,model_load_s=MODEL_LOAD_S,embedding_prep_s=EMBED_PREP_S,embedding_bytes=doc_emb.nbytes,tfidf_bytes=doc_tf.data.nbytes+doc_tf.indices.nbytes+doc_tf.indptr.nbytes,packages={p:importlib.metadata.version(p) for p in ['sentence-transformers','scikit-learn','torch','numpy','pandas']},dataset_sha256={n:hashlib.sha256(Path(n+'.csv').read_bytes()).hexdigest() for n in ['records','queries','labels']})
cur.execute('SELECT version()');metadata['postgresql']=cur.fetchone()[0]
(output/'metadata.json').write_text(json.dumps(metadata,indent=2))
(output/'report.md').write_text('# Synthetic duplicate-review pilot\n\n'+summary.to_markdown(index=False)+'\n\nSynthetic template labels only; no real-world accuracy claim. A/B/C use identical trigram candidates. D unions up to K trigram and K semantic candidates. Exact normalized-title matches bypass thresholds in all methods. Thresholds are tuned on development pairs. Recall@5 is measured before thresholding; alert recall is measured after thresholding. CPU warm timings include query vectorization but exclude offline preparation. Dataset is too small for GIN scalability conclusions. Confidence intervals resample queries, not independent real-world studies. Related studies are negative controls, not automatically duplicate records.\n')
archive='/content/duplicate_benchmark_results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in output.iterdir():z.write(p,p.name)
files.download(archive)
